# 05 — Fine-Tune the LLM (QLoRA via Unsloth)

**Project:** Ecommerce-LLM-Finetuning
**Goal of this notebook (the core of the project):**
1. Install Colab-pinned dependencies (Unsloth + friends)
2. Detect and confirm GPU (Colab Free T4)
3. Load the base model in 4-bit (QLoRA)
4. Attach a LoRA adapter
5. Load the formatted train/val datasets from notebook 04
6. Configure and run the `SFTTrainer`
7. Plot training/validation loss
8. Save the LoRA adapter + tokenizer

**Runtime:** Google Colab Free, T4 GPU. Runtime > Change runtime type > **T4 GPU**.

⚠️ If you see an OOM (out-of-memory) error, reduce `per_device_train_batch_size`
in `src/config.py` (e.g. to 1) and/or increase `gradient_accumulation_steps`.


## 1. Install dependencies

Unsloth's installer is version-sensitive to the Colab CUDA/torch build, so we use their official Colab-safe install command rather than plain `pip install unsloth`.

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
%%capture
# Colab-safe Unsloth install (handles the correct xformers/torch/CUDA combo)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install -q datasets sentencepiece


## 2. GPU detection

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.utils import detect_device, supports_bf16, get_logger

logger = get_logger("notebook05")
device = detect_device()
print(f"Device: {device}")
print(f"bf16 supported: {supports_bf16()}")

if device != "cuda":
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > Hardware "
        "accelerator > T4 GPU, then re-run this notebook from the top."
    )

!nvidia-smi


## 3. Load config and formatted datasets

In [ ]:
from src.config import Config
from src.utils import load_jsonl
from datasets import Dataset

cfg = Config()
cfg.ensure_dirs()

train_records = load_jsonl(cfg.processed_data_dir / "train.jsonl")
val_records = load_jsonl(cfg.processed_data_dir / "val.jsonl")

train_dataset = Dataset.from_list(train_records)
val_dataset = Dataset.from_list(val_records)

print(train_dataset)
print(val_dataset)
print("\nSample training text:\n")
print(train_dataset[0]["text"])


## 4. Load base model in 4-bit (QLoRA) + attach LoRA adapter

In [ ]:
from src.trainer import EcommerceFineTuner

fine_tuner = EcommerceFineTuner(cfg)
model, tokenizer = fine_tuner.load_base_model()
model = fine_tuner.apply_lora()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")


## 5. Configure and run the SFTTrainer

Uses:
- `max_seq_length` from config
- gradient checkpointing (Unsloth's optimized variant)
- bf16 on Ampere+/T4-supported GPUs, fp16 fallback otherwise
- gradient accumulation for a larger effective batch size
- linear LR schedule with warmup
- periodic evaluation on the validation split

In [ ]:
trainer = fine_tuner.build_trainer(train_dataset, val_dataset)
print(trainer.args)


In [ ]:
train_result = fine_tuner.train()
train_result.metrics


## 6. Plot training & validation loss

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = fine_tuner.get_loss_history()
train_hist = history[history["loss"].notna()][["step", "loss"]] if "step" in history.columns else history[history["loss"].notna()][["loss"]].reset_index()
eval_hist = history[history.get("eval_loss").notna()] if "eval_loss" in history.columns else pd.DataFrame()

plt.figure(figsize=(9, 5))
if "step" in history.columns:
    plt.plot(train_hist["step"], train_hist["loss"], label="train_loss", marker="o", markersize=3)
    if not eval_hist.empty:
        plt.plot(eval_hist["step"], eval_hist["eval_loss"], label="val_loss", marker="s", markersize=4)
else:
    plt.plot(train_hist["loss"].values, label="train_loss")

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 7. Save the LoRA adapter + tokenizer

In [ ]:
save_dir = fine_tuner.save_adapter()
print(f"Adapter saved to: {save_dir}")
!ls -lh {save_dir}


## 8. Quick smoke test

Generate one response right away (before formal evaluation in notebook 06) to sanity-check the fine-tune worked.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

test_instruction = "Where is my order? It's been 5 days."
prompt = f"{cfg.instruction_header}\nCustomer:\n{test_instruction}\n\n{cfg.response_header}\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=120, temperature=0.7, do_sample=True)
print(tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))


## Summary

- Loaded `Config.base_model_name` in 4-bit and attached a LoRA adapter (r=16, targeting attention + MLP projections)
- Trained with `SFTTrainer` using gradient accumulation, warmup, and bf16/fp16 auto-selection
- Plotted train/validation loss to confirm convergence
- Saved the lightweight LoRA adapter to `models/lora_adapter/`

Next: `06_evaluation.ipynb` for BLEU/ROUGE/perplexity scoring, then `07_inference.ipynb` to merge and deploy.